# PyTorch Dataset for AnnData Objects

**Author:** AnnData Development Team

This tutorial shows how to use `AnnDataset` with real single-cell data to build PyTorch models with custom transforms and multiprocessing.


In [1]:
import numpy as np
import scanpy as sc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

from anndata.experimental.pytorch import AnnDataset


## Load real single-cell data


In [2]:
# Load PBMC 3k dataset (a classic single-cell dataset)
adata = sc.datasets.pbmc3k_processed()

print(f"Loaded: {adata}")
print(f"Cell types: {adata.obs['louvain'].value_counts()[:5]}")
print(f"Data is sparse: {adata.X.__class__.__name__}")


Loaded: AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
Cell types: louvain
CD4 T cells        1144
CD14+ Monocytes     480
B cells             342
CD8 T cells         316
NK cells            154
Name: count, dtype: int64
Data is sparse: ndarray


## Create custom transforms for ML training


In [3]:
# Select top variable genes for training
print("Selecting top 2000 variable genes...")
gene_var = np.var(adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X, axis=0)
top_genes = np.argsort(gene_var)[-2000:]  # Top 2000 most variable genes
adata_subset = adata[:, top_genes].copy()

print(f"Reduced from {adata.n_vars} to {adata_subset.n_vars} genes")

# Define transform functions for training
def training_transform(X):
    """Normalize and add augmentation for training."""
    # Ensure we have positive values and no NaNs
    X = torch.clamp(X, min=0)

    # Normalize to 10k counts per cell (sum across genes for each cell)
    row_sum = torch.sum(X, dim=-1, keepdim=True) + 1e-8  # Sum across genes (last dimension)
    X = X * (1e4 / row_sum)

    # Log transform
    X = torch.log1p(X)

    # Add small amount of noise for augmentation
    if torch.rand(1).item() < 0.5:  # 50% chance
        noise = torch.normal(0, 0.01, size=X.shape)  # Reduced noise
        X = X + noise

    # Ensure no NaNs or infinities
    X = torch.nan_to_num(X, nan=0.0, posinf=10.0, neginf=0.0)

    return X

print("Created transform function")


Selecting top 2000 variable genes...
Reduced from 1838 to 1838 genes
Created transform function


## Create dataset and dataloader with multiprocessing


In [4]:
# Create training dataset with preprocessed data
dataset = AnnDataset(
    adata_subset,  # Use gene-selected data
    transform=training_transform,
    multiprocessing_safe=True
)

# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print(f"Dataset: {len(dataset)} cells")
print(f"DataLoader: {len(dataloader)} batches")

# Test a batch and check for issues
batch = next(iter(dataloader))
print(f"Batch shape: {batch['X'].shape}")
print(f"Batch range: [{batch['X'].min():.2f}, {batch['X'].max():.2f}]")
print(f"Contains NaN: {torch.isnan(batch['X']).any().item()}")
print(f"Contains Inf: {torch.isinf(batch['X']).any().item()}")

# Note: For multiprocessing in production, define transforms in a separate module


Dataset: 2638 cells
DataLoader: 21 batches
Batch shape: torch.Size([128, 1838])
Batch range: [-0.05, 6.12]
Contains NaN: False
Contains Inf: False


## DataLoader integration


In [5]:
class SimpleAutoencoder(nn.Module):
    """Simple autoencoder for dimensionality reduction."""
    def __init__(self, input_dim: int, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, input_dim),
            nn.ReLU()  # Non-negative output for gene expression
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Initialize model
input_dim = batch['X'].shape[1]  # Number of genes after selection
model = SimpleAutoencoder(input_dim, latent_dim=32)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print(f"Model input dimension: {input_dim}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model input dimension: 1838
Model parameters: 2,164,046


## Custom transforms


In [6]:
# Training loop
model.train()
n_epochs = 3

for epoch in range(n_epochs):
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(dataloader):
        # Get data (multiprocessing automatically handles transforms)
        X = batch['X'].float()

        # Forward pass
        optimizer.zero_grad()
        reconstructed = model(X)
        loss = criterion(reconstructed, X)

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        # Print progress
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / n_batches
    print(f"Epoch {epoch+1} completed. Average loss: {avg_loss:.4f}\n")

print("Training completed!")


Epoch 1/3, Batch 0/21, Loss: 1.3372
Epoch 1/3, Batch 10/21, Loss: 1.1473
Epoch 1/3, Batch 20/21, Loss: 1.0931
Epoch 1 completed. Average loss: 1.1683

Epoch 2/3, Batch 0/21, Loss: 1.1143
Epoch 2/3, Batch 10/21, Loss: 1.1431
Epoch 2/3, Batch 20/21, Loss: 1.1055
Epoch 2 completed. Average loss: 1.1168

Epoch 3/3, Batch 0/21, Loss: 1.0868
Epoch 3/3, Batch 10/21, Loss: 1.0860
Epoch 3/3, Batch 20/21, Loss: 1.0861
Epoch 3 completed. Average loss: 1.0926

Training completed!


## Multiprocessing Note

**Important**: Functions defined in Jupyter notebooks can't be pickled for multiprocessing. In production code, define your transforms in a separate Python module:

```python
# In transforms.py
def training_transform(X):
    row_sum = torch.sum(X, dim=0, keepdim=True) + 1e-8
    X = X * (1e4 / row_sum)
    return torch.log1p(X)

# In your main script
from transforms import training_transform
dataloader = DataLoader(dataset, num_workers=4)  # Now multiprocessing works!
```


In [7]:
# Create evaluation dataset (without augmentation)
def eval_transform(X):
    """Normalize and log transform for evaluation (no augmentation)."""
    # Ensure we have positive values
    X = torch.clamp(X, min=0)

    # Normalize to 10k counts per cell (sum across genes for each cell)
    row_sum = torch.sum(X, dim=-1, keepdim=True) + 1e-8
    X = X * (1e4 / row_sum)

    # Log transform
    X = torch.log1p(X)

    # Ensure no NaNs or infinities
    X = torch.nan_to_num(X, nan=0.0, posinf=10.0, neginf=0.0)

    return X

eval_dataset = AnnDataset(adata_subset, transform=eval_transform, multiprocessing_safe=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=256, num_workers=0, pin_memory=False)

# Extract embeddings
model.eval()
embeddings = []
reconstruction_losses = []

with torch.no_grad():
    for batch in eval_dataloader:
        X = batch['X'].float()

        # Get latent embeddings
        encoded = model.encoder(X)
        embeddings.append(encoded)

        # Calculate reconstruction loss
        reconstructed = model(X)
        loss = criterion(reconstructed, X)
        reconstruction_losses.append(loss.item())

# Concatenate all embeddings
embeddings = torch.cat(embeddings, dim=0).numpy()
avg_reconstruction_loss = np.mean(reconstruction_losses)

print(f"Extracted embeddings shape: {embeddings.shape}")
print(f"Average reconstruction loss: {avg_reconstruction_loss:.4f}")

# Add embeddings to AnnData object
adata.obsm['X_autoencoder'] = embeddings
print("\nEmbeddings added to adata.obsm['X_autoencoder']")
print(f"Final AnnData object: {adata}")


Extracted embeddings shape: (2638, 32)
Average reconstruction loss: 1.0804

Embeddings added to adata.obsm['X_autoencoder']
Final AnnData object: AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr', 'X_autoencoder'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
